# **Midterm Project**

## Data Preparation

In [146]:
import pandas as pd

# Load daily stock data to dataframes
universe = ['AAPL','AMZN','GOOG','IBM','META','MSFT','NFLX','ORCL','SAP','TSLA']
data = {}

for ticker in universe:
    df = pd.read_csv(f'{ticker}.csv', index_col=0)
    df.index = pd.to_datetime(df.index, format='%d-%b-%y', errors='coerce')
    data[ticker] = df

## Close Data

In [147]:
# Extract close and adj close data
close_data = pd.DataFrame()

for ticker, df in data.items():
    close_data[f'{ticker} Close'] = df['Close']
    close_data[f'{ticker} Adj Close'] = df['Adj Close']

# Extra precaution
close_data.sort_index(inplace=True)

print(close_data.head())

            AAPL Close  AAPL Adj Close  AMZN Close  AMZN Adj Close  \
Date                                                                 
2018-01-02       43.06           40.57       59.45           59.45   
2018-01-03       43.06           40.56       60.21           60.21   
2018-01-04       43.26           40.75       60.48           60.48   
2018-01-05       43.75           41.21       61.46           61.46   
2018-01-08       43.59           41.06       62.34           62.34   

            GOOG Close  GOOG Adj Close  IBM Close  IBM Adj Close  META Close  \
Date                                                                           
2018-01-02       53.25           53.12     147.47         107.53      181.42   
2018-01-03       54.12           53.99     151.52         110.49      184.67   
2018-01-04       54.32           54.19     154.59         112.72      184.33   
2018-01-05       55.11           54.98     155.34         113.27      186.85   
2018-01-08       55.35       

## 1.1 Portfolio set-up

In [148]:
class Portfolio:
    def __init__(self, tickers, initial_cash=5e6, start_date='2018-01-01'):
        self.cash = initial_cash
        self.tickers = tickers
        self.start_date = pd.to_datetime(start_date)
        
        # each ticker has its own transaction DataFrame
        self.holdings = {
            t: pd.DataFrame([{
                'Buy/Sell': 0,
                'Stock Price': 0.0,
                'Delta Shares': 0.0,
                'Current Holdings': 0,
                'Current Value': 0.0
            }], index=[self.start_date])
            for t in tickers
        }

    @staticmethod
    def _closest_trading_date(date, price_data):
        """Return the closest previous trading date in price_data if date is missing."""
        date = pd.to_datetime(date)
        if date not in price_data.index:
            # method='ffill' finds previous available date
            idx = price_data.index.get_indexer([date], method='ffill')[0]
            date = price_data.index[idx]
        return date

    def buy(self, ticker, trade_date, price, shares):
        cost = price * shares
        if cost > self.cash:
            print(f"⚠️ Not enough cash to buy {shares} shares of {ticker}")
            return
        self.cash -= cost
        prev = self.holdings[ticker].iloc[-1]['Current Holdings']
        new = prev + shares
        self.holdings[ticker].loc[pd.to_datetime(trade_date)] = {
            'Buy/Sell': 1,
            'Stock Price': price,
            'Delta Shares': shares,
            'Current Holdings': new,
            'Current Value': new * price
        }

    def sell(self, ticker, trade_date, price, shares):
        prev = self.holdings[ticker].iloc[-1]['Current Holdings']
        if shares > prev:
            print(f"⚠️ Not enough shares to sell {shares} of {ticker}")
            return
        proceeds = price * shares
        self.cash += proceeds
        new = prev - shares
        self.holdings[ticker].loc[pd.to_datetime(trade_date)] = {
            'Buy/Sell': 0,
            'Stock Price': price,
            'Delta Shares': -shares,
            'Current Holdings': new,
            'Current Value': new * price
        }

    def liquidate(self, date, close_data):
        """
        Sell all current holdings of all tickers at 'Close' prices on the given date.
        Automatically picks the closest previous trading day if date is not available.
        """
        date = self._closest_trading_date(date, close_data)
        for ticker in self.tickers:
            shares = self.holdings[ticker].iloc[-1]['Current Holdings']
            if shares > 0:
                price = close_data.loc[date, f'{ticker} Close']
                self.sell(ticker, date, price, shares)

    def get_value(self, prices, date):
        """Compute total mark-to-market value given price data and date."""
        date = self._closest_trading_date(date, prices)
        total_value = self.cash
        for t in self.tickers:
            px = prices.loc[date, f'{t} Close']
            shares = self.holdings[t].iloc[-1]['Current Holdings']
            total_value += shares * px
        return total_value

    def mtm(self):
        """
        Compute total mark-to-market portfolio value using the last known
        stock price for each holding (from the portfolio DataFrames) + cash.
        """
        total_value = self.cash
        for ticker in self.tickers:
            df = self.holdings[ticker]
            if not df.empty:
                last_row = df.iloc[-1]
                shares = last_row['Current Holdings']
                price = last_row['Stock Price']
                total_value += shares * price
        return total_value


## 1.2 Initial trades

In [149]:
pyport = Portfolio(universe)
allocation = 1e6
buy_list = ['IBM','MSFT','GOOG','AAPL','AMZN']
trade_date = '2018-01-02'

for ticker in buy_list:
    price = close_data.loc[trade_date, f'{ticker} Close']
    shares = int(allocation // price)
    pyport.buy(ticker, trade_date, price, shares)

print(f"Remaining cash: ${pyport.cash:,.2f}")
for t in pyport.tickers:
    print(f"{t} holdings: {pyport.holdings[t].iloc[-1]['Current Holdings']}")

Remaining cash: $150.50
AAPL holdings: 23223.0
AMZN holdings: 16820.0
GOOG holdings: 18779.0
IBM holdings: 6781.0
META holdings: 0.0
MSFT holdings: 11634.0
NFLX holdings: 0.0
ORCL holdings: 0.0
SAP holdings: 0.0
TSLA holdings: 0.0


## 2.1 Rebalancing low strategy

In [150]:
def rebal_low(date, adj_close_data, top_n=5):
    """
    Find the top_n stocks that dropped the most in Adj Close over the past 5 business days.
    Automatically adjusts for holidays/weekends by using the closest previous available date.
    """
    date = pd.to_datetime(date)

    # Find the closest previous available date in the index
    if date not in adj_close_data.index:
        date = adj_close_data.index[adj_close_data.index.get_indexer([date], method='ffill')[0]]

    prev_date_idx = adj_close_data.index.get_loc(date) - 5
    if prev_date_idx < 0:
        raise ValueError("Not enough historical data for 5-day drop calculation.")
    
    prev_date = adj_close_data.index[prev_date_idx]

    # Compute percentage change over the 5 days
    pct_change = {}
    for ticker in universe:
        start_price = adj_close_data.loc[prev_date, f'{ticker} Adj Close']
        end_price = adj_close_data.loc[date, f'{ticker} Adj Close']
        pct_change[ticker] = (end_price - start_price) / start_price

    # Sort by percentage drop (most negative first)
    sorted_drop = sorted(pct_change.items(), key=lambda x: x[1])
    
    # Return the tickers of the top_n largest drops
    tickers_to_buy = [ticker for ticker, change in sorted_drop[:top_n]]
    
    return tickers_to_buy


## 2.2 Five day increments

In [ ]:
start_date = pd.to_datetime('2018-01-09')
transaction_days = close_data.index[close_data.index >= start_date][::5]

DatetimeIndex(['2018-01-09', '2018-01-17', '2018-01-24', '2018-01-31',
               '2018-02-07', '2018-02-14', '2018-02-22', '2018-03-01',
               '2018-03-08', '2018-03-15'],
              dtype='datetime64[ns]', name='Date', freq=None)


In [151]:
# Liquidate entire portfolio on Jan 09 2018
rebalance_date = '2018-01-09'
pyport.liquidate(rebalance_date, close_data)

In [152]:
rebalance_date = '2018-01-15'
tickers_to_buy = rebal_low(rebalance_date, close_data)
print("Top 5 dropped stocks:", tickers_to_buy)


Top 5 dropped stocks: ['SAP', 'META', 'IBM', 'AAPL', 'MSFT']


In [153]:
rebalance_date = '2018-01-15'  # example rebalance date

# Make sure the date exists in close_data
rebalance_date = pyport._closest_trading_date(rebalance_date, close_data)

# Split cash equally among top 5 tickers
allocation_per_stock = pyport.cash / len(tickers_to_buy)

for ticker in tickers_to_buy:
    # Get the Close price for the ticker on the rebalance date
    price = close_data.loc[rebalance_date, f'{ticker} Close']
    
    # Calculate the maximum number of shares you can buy with your allocation
    shares = int(allocation_per_stock // price)
    
    # Buy the shares
    pyport.buy(ticker, rebalance_date, price, shares)


In [154]:
# Transaction history
for t in universe:
    print(t)
    print(pyport.holdings[t])
    print()

AAPL
            Buy/Sell  Stock Price  Delta Shares  Current Holdings  \
2018-01-01         0         0.00           0.0               0.0   
2018-01-02         1        43.06       23223.0           23223.0   
2018-01-09         0        43.58      -23223.0               0.0   
2018-01-12         1        44.27       23459.0           23459.0   

            Current Value  
2018-01-01           0.00  
2018-01-02      999982.38  
2018-01-09           0.00  
2018-01-12     1038529.93  

AMZN
            Buy/Sell  Stock Price  Delta Shares  Current Holdings  \
2018-01-01         0         0.00           0.0               0.0   
2018-01-02         1        59.45       16820.0           16820.0   
2018-01-09         0        62.63      -16820.0               0.0   

            Current Value  
2018-01-01            0.0  
2018-01-02       999949.0  
2018-01-09            0.0  

GOOG
            Buy/Sell  Stock Price  Delta Shares  Current Holdings  \
2018-01-01         0         0.00      